# HAKE-MER — export F1 par classe (test)

Ré-entraîne les campagnes **Step~0, M1, M1+M2** (DistilBERT, 3 graines) avec le code qui enregistre `test.per_label` dans chaque `metrics.json`, puis agrège `reference/artifacts/per_class_ladder_distilbert_seeds_42_123_456.json`.

**Durée indicative:** ~2 h GPU (9 entraînements, même protocole que ch.~4).

Télécharger le zip final + cette `.ipynb` exécutée vers `reference/training_records/step_per_class/colab/`.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
e5fa7e1


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.4 MB/s eta 0:00:00


In [4]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}
!./run_m1_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}
!./run_m1_m2_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 1.99MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 233kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.31MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.36MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 26.8MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:  58% 1.60M/2.77M [00:00<00:00, 2.58MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 3.46MB/s,  265kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 3.51MB/s,  270kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 639kB/s, 34.1kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 647kB/s, 34.6kB/s  ]

simplified/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/347k [

In [5]:
!./run_per_class_from_metrics.sh --seeds 42,123,456

skip (no metrics): runs/distilbert_base_uncased_seed42_m1_m2_m3_nrc/metrics.json
skip (no metrics): runs/distilbert_base_uncased_seed123_m1_m2_m3_nrc/metrics.json
skip (no metrics): runs/distilbert_base_uncased_seed456_m1_m2_m3_nrc/metrics.json
skip (no metrics): runs/distilbert_base_uncased_seed42_m1_m2_m3_senticnet/metrics.json
skip (no metrics): runs/distilbert_base_uncased_seed123_m1_m2_m3_senticnet/metrics.json
skip (no metrics): runs/distilbert_base_uncased_seed456_m1_m2_m3_senticnet/metrics.json
Wrote reference/artifacts/per_class_ladder_distilbert_seeds_42_123_456.json (3 configurations)


In [6]:
import json
import zipfile
from pathlib import Path
from google.colab import files

artifact = Path("reference/artifacts/per_class_ladder_distilbert_seeds_42_123_456.json")
c = json.loads(artifact.read_text(encoding="utf-8"))
print("configurations:", len(c["configurations"]))
for cfg in c["configurations"]:
    print(cfg["configuration"], "macro", f"{cfg['test_f1_macro_mean']:.4f}", "seeds", cfg["seeds_evaluated"])

zip_path = Path("/content/per_class_ladder_distilbert_seeds_42_123_456.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(artifact, artifact.name)
    for p in sorted(Path("runs").glob("distilbert_base_uncased_seed*_*/metrics.json")):
        if any(x in p.parent.name for x in ("baseline_plm", "_m1", "_m1_m2")) and "m3" not in p.parent.name:
            zf.write(p, f"{p.parent.name}/{p.name}")
files.download(str(zip_path))

configurations: 3
Step~0 PLM (flat) macro 0.4876 seeds [42, 123, 456]
+M1 macro 0.4581 seeds [42, 123, 456]
+M1+M2 macro 0.5050 seeds [42, 123, 456]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>